# State Management with StateMachine
This notebook demonstrates how to work with state management using a StateMachine implementation. We'll explore how to create, manage, and control workflow states in a structured way.

## What we'll learn:
- Basic state machine concepts and implementation
- Creating and connecting workflow steps
- Managing state transitions and data flow
- Working with routing and loops in state machines
- Understanding state snapshots and execution flow

### Setup

In [44]:
from typing import TypedDict
from lib.state_machine import (
    StateMachine,
    Step,
    EntryPoint,
    Termination,
)

## Basic State Machine Concepts
Let's start with a simple example that demonstrates the core concepts of our state machine:
1. Defining state schema
2. Creating steps
3. Connecting steps
4. Running the workflow

**Creating the Schema and the State Machine**

In [45]:
class Schema(TypedDict):
    """Schema defining the structure of our state.
    
    Attributes:
        input: The input value to process
        output: The processed output value
    """
    input: int
    output: int

In [46]:
# Create our state machine instance
workflow = StateMachine(Schema)

**Defining the logic for Steps**

In [47]:
def step_input(state: Schema) -> Schema:
    """First step: Increment the input value.
    
    Args:
        state: Current state containing input value
        
    Returns:
        Updated state with incremented value in output
    """
    return {"output": state["input"] + 1, "random": 10}



In [48]:
def step_double(state: Schema) -> Schema:
    """Second step: Double the previous output.
    
    Args:
        state: Current state containing output from previous step
        
    Returns:
        Updated state with doubled output value
    """
    return {"output": state["output"] * 2}


**Creating and Connecting Steps**

In [49]:
entry = EntryPoint()
s1 = Step("input", step_input)
s2 = Step("double", step_double)
termination = Termination()

In [50]:
workflow.add_steps([entry, s1, s2, termination])

In [51]:
workflow.connect(entry, s1)
workflow.connect(s1, s2)
workflow.connect(s2, termination)

In [52]:
workflow.transitions

{'__entry__': [Transition('__entry__' -> ['input'])],
 'input': [Transition('input' -> ['double'])],
 'double': [Transition('double' -> ['__termination__'])]}

**Running the Workflow**

In [53]:
initial_state = {"input": 4}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: __entry__
[StateMachine] Executing step: input
[StateMachine] Executing step: double
[StateMachine] Terminating: __termination__


Run('7da2a934-9ab6-4561-9c6e-0fa62d7b36f7')

In [54]:
run_object.snapshots

[Snapshot('9904506c-a89a-4b11-b861-5735cb393398') @ [2026-03-11 12:55:43.697231]: __entry__.State({'input': 4}),
 Snapshot('cf830898-2bc0-4ecc-9667-7e9527049b1b') @ [2026-03-11 12:55:43.697277]: input.State({'input': 4, 'output': 5}),
 Snapshot('d9abd30e-408e-4f11-9d04-0283db31447a') @ [2026-03-11 12:55:43.697508]: double.State({'input': 4, 'output': 10})]

## Advanced State Management: Routing and Loops
Now we'll explore more complex state management patterns including:
- Conditional routing between steps
- Creating loops in the workflow
- Managing state through multiple iterations

In [55]:
class CounterSchema(TypedDict):
    """Schema for a counter-based workflow.
    
    Attributes:
        count: Current counter value
        max_value: Maximum value before termination
    """
    count: int
    max_value: int

In [56]:
workflow = StateMachine(CounterSchema)

In [57]:
def increment_counter(state: CounterSchema) -> CounterSchema:
    """Increment the counter value.
    
    Args:
        state: Current state with counter value
        
    Returns:
        Updated state with incremented counter
    """
    return {"count": state["count"] + 1}

In [58]:
# Create steps
entry = EntryPoint()
increment = Step("increment", increment_counter)
termination = Termination()

In [59]:
workflow.add_steps([entry, increment, termination])

In [60]:
# Router logic
def check_counter(state: CounterSchema) -> Step:
    """Determine next step based on counter value.
    
    Args:
        state: Current state with counter and max value
        
    Returns:
        Next step to execute (increment or terminate)
    """
    if state["count"] >= state["max_value"]:
        return termination
    return increment

In [61]:
# Connect steps with a loop in increment
workflow.connect(entry, increment)
workflow.connect(increment, [increment, termination], check_counter)

In [62]:
workflow.transitions

{'__entry__': [Transition('__entry__' -> ['increment'])],
 'increment': [Transition('increment' -> ['increment', '__termination__'])]}

In [63]:
initial_state = {"count": 0, "max_value": 3}
run_object = workflow.run(initial_state)
run_object

[StateMachine] Starting: __entry__
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Executing step: increment
[StateMachine] Terminating: __termination__


Run('129a3d1d-6ea6-45f5-a42c-e7c1ef31681e')

In [64]:
run_object.snapshots

[Snapshot('2e234c40-6873-47ed-9e2e-21a87d77745e') @ [2026-03-11 12:55:43.789845]: __entry__.State({'count': 0, 'max_value': 3}),
 Snapshot('d631657e-cf2e-467f-8d54-ae49b86cca3d') @ [2026-03-11 12:55:43.790195]: increment.State({'count': 1, 'max_value': 3}),
 Snapshot('de932411-257d-4b6f-be50-1fc5e906b189') @ [2026-03-11 12:55:43.790350]: increment.State({'count': 2, 'max_value': 3}),
 Snapshot('fa77ab75-03ae-4e3a-ba87-7dd4117892f4') @ [2026-03-11 12:55:43.790394]: increment.State({'count': 3, 'max_value': 3})]

In [65]:
run_object.get_final_state()

{'count': 3, 'max_value': 3}